![Universidad Iberoamericana del Ecuador](https://raw.githubusercontent.com/andresrubiop/ml-unibe-estudiantes/main/utils/logo-unibe-color.png)

<div style="background:#004890;color:#FFFFFF;padding:14px 18px;border-radius:10px;margin:6px 0 12px 0;border-bottom:4px solid #FCC000"><div style="font-size:12px;letter-spacing:.08em;text-transform:uppercase;opacity:.9">Machine Learning · Sesión 1 · Notebook del estudiante · no calificable</div><div style="font-size:22px;font-weight:700;margin-top:4px">L1.2 · MNIST con seis clasificadores</div><div style="font-size:12px;opacity:.9;margin-top:4px">Posgrado · Maestría en Big Data y Ciencia de Datos · UNIB.E</div></div>

| | |
|---|---|
| **Objetivo** | Resolver el problema icónico de clasificación, reconocer dígitos escritos a mano, con los seis clasificadores del programa y compararlos con honestidad. |
| **Duración** | ≈ 90 min con el reto · en el taller de la **Clase 1** (35 min) se completan las celdas `# TODO` de las partes 1 a 5 |
| **Teoría** | Manual M1 §4–§8 · Animaciones A1.5 a A1.10 |
| **Datos icónicos** | **dígitos de scikit-learn** (1 797 imágenes de 8 × 8) y **MNIST** (70 000 imágenes de 28 × 28; LeCun, Cortes y Burges) |
| **Escala realista (reto)** | **Fashion-MNIST** (70 000 fotos de prendas de ropa de Zalando; Xiao y colaboradores, 2017) |

**Partes:** 1 · conocer los datos → 2 · regresión logística multinomial (softmax) → 3 · árbol de decisión → 4 · modelos generativos: LDA, QDA y Naive Bayes → 5 · SVM y k vecinos, y la tabla final → reto con Fashion-MNIST.

MNIST se descarga una sola vez (11 MB) desde el repositorio público de Keras y queda en caché. Para que todo corra en pocos minutos usamos **10 000 imágenes de entrenamiento** y **2 000 de prueba**.

## 0 · Configuración

In [ ]:
import gzip
import os
import time
import urllib.request
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from cycler import cycler

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 2026
UNIBE = {"azul": "#004890", "naranja": "#C2410C", "dorado": "#A87900", "verde": "#3A7D44", "gris": "#77787B"}
plt.rcParams.update({
    "axes.prop_cycle": cycler(color=list(UNIBE.values())), "axes.titlecolor": UNIBE["azul"],
    "axes.titleweight": "bold", "axes.edgecolor": UNIBE["gris"], "axes.grid": True, "grid.color": "#E8EDF2",
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110, "legend.frameon": False,
})
CACHE = os.path.expanduser("~/.cache/ml-unibe")
URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/"


def descargar(nombre):
    """Descarga un archivo del repositorio público de datasets de Keras (una sola vez)."""
    os.makedirs(CACHE, exist_ok=True)
    ruta = os.path.join(CACHE, nombre)
    if not os.path.exists(ruta):
        urllib.request.urlretrieve(URL + nombre, ruta)
    return ruta


def cargar_mnist():
    """MNIST: 60 000 imágenes de entrenamiento y 10 000 de prueba, 28 × 28 píxeles con valores de 0 a 255."""
    d = np.load(descargar("mnist.npz"))
    return d["x_train"], d["y_train"], d["x_test"], d["y_test"]


print(f"scikit-learn {sklearn.__version__} · semilla {SEED}")

## Parte 1 · Conocer los datos

Empezamos con la versión de juguete que trae scikit-learn: 1 797 dígitos de 8 × 8 píxeles. Cada imagen es una **fila** y cada píxel una **variable**: 64 columnas con la intensidad de gris.

In [ ]:
from sklearn.datasets import load_digits

dig = load_digits()
fig, axs = plt.subplots(1, 10, figsize=(10, 1.4))
for k, ax in enumerate(axs):
    ax.imshow(dig.images[k], cmap="Greys"); ax.set_title(dig.target[k]); ax.axis("off")
plt.suptitle("Dígitos de scikit-learn (8 × 8)", y=1.08); plt.show()
print("Matriz de datos:", dig.data.shape, "· la primera imagen como fila de 64 números:\n", dig.data[0].astype(int))

Ahora el MNIST de verdad: 28 × 28 = **784 variables** por imagen.

In [ ]:
Xtr_full, ytr_full, Xte_full, yte_full = cargar_mnist()
print("Entrenamiento:", Xtr_full.shape, "· prueba:", Xte_full.shape)
rng = np.random.default_rng(SEED)
idx_tr = rng.choice(len(Xtr_full), 10_000, replace=False)
idx_te = rng.choice(len(Xte_full), 2_000, replace=False)
X_tr, y_tr = Xtr_full[idx_tr].reshape(-1, 784) / 255.0, ytr_full[idx_tr]   # escalamos a [0, 1]
X_te, y_te = Xte_full[idx_te].reshape(-1, 784) / 255.0, yte_full[idx_te]
fig, axs = plt.subplots(2, 10, figsize=(10, 2.4))
for k, ax in enumerate(axs.ravel()):
    ax.imshow(X_tr[k].reshape(28, 28), cmap="Greys"); ax.set_title(y_tr[k], fontsize=9); ax.axis("off")
plt.suptitle("MNIST (28 × 28)", y=1.02); plt.show()

**Tu turno.** ¿Están balanceadas las clases? Calcula la proporción de cada dígito en `y_tr` y el **baseline**: la exactitud en la prueba de predecir siempre el dígito más frecuente del entrenamiento.

In [ ]:
# TODO: calcula la proporción por dígito (np.bincount) y la exactitud del baseline en la prueba.
prop = ...
mas_frecuente = ...
acc_base = ...
print("Proporciones:", prop.round(3), "· más frecuente:", mas_frecuente, f"· baseline = {acc_base:.3f}")
assert 0.08 < acc_base < 0.14, "Con 10 clases casi balanceadas, el baseline ronda el 10–12 %."

Para comparar los seis modelos guardamos su exactitud y su tiempo en una tabla.

In [ ]:
resultados = []


def evaluar(nombre, modelo):
    t0 = time.perf_counter(); modelo.fit(X_tr, y_tr); t1 = time.perf_counter()
    acc = modelo.score(X_te, y_te); t2 = time.perf_counter()
    resultados.append({"modelo": nombre, "exactitud": acc, "entrenar (s)": t1 - t0, "predecir (s)": t2 - t1})
    print(f"{nombre:<34} exactitud = {acc:.3f} · entrenar {t1 - t0:5.1f} s · predecir {t2 - t1:5.2f} s")
    return modelo

## Parte 2 · Regresión logística multinomial (softmax)

Diez puntajes, uno por dígito, convertidos en probabilidades con softmax (M1 §4). Cada clase tiene un vector de 784 pesos: **uno por píxel**.

In [ ]:
from sklearn.linear_model import LogisticRegression

softmax = evaluar("Logística multinomial (softmax)", LogisticRegression(max_iter=300, C=0.1))
fig, axs = plt.subplots(1, 10, figsize=(11, 1.6))
lim = np.abs(softmax.coef_).max()
for k, ax in enumerate(axs):
    ax.imshow(softmax.coef_[k].reshape(28, 28), cmap="RdBu", vmin=-lim, vmax=lim); ax.set_title(k); ax.axis("off")
plt.suptitle("Pesos de cada clase: azul empuja hacia el dígito, rojo en contra", y=1.1); plt.show()

**Tu turno.** Toma la primera imagen de la prueba y muestra sus 10 probabilidades. ¿Suman 1? ¿Qué tan seguro está el modelo?

In [ ]:
# TODO: calcula las 10 probabilidades de X_te[:1] con softmax.predict_proba.
p0 = ...
print("Dígito real:", y_te[0], "· probabilidades:", p0.round(3), "· suma:", round(float(p0.sum()), 6))
assert abs(p0.sum() - 1) < 1e-9

## Parte 3 · Árbol de decisión

Un árbol pregunta por píxeles: «¿el píxel 350 es más oscuro que 0.3?». Veamos cómo cambia con la profundidad.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

for prof in (2, 5, 10, 20):
    t = DecisionTreeClassifier(max_depth=prof, random_state=SEED).fit(X_tr, y_tr)
    print(f"profundidad {prof:>2}: entrenamiento = {t.score(X_tr, y_tr):.3f} · prueba = {t.score(X_te, y_te):.3f} · hojas = {t.get_n_leaves()}")

**Tu turno.** Elige la profundidad que te parezca mejor según lo anterior y evalúa ese árbol con `evaluar`. (Ojo: elegir con la prueba es una pequeña trampa que corregiremos en L1.3 con validación cruzada.)

In [ ]:
# TODO: evalúa un DecisionTreeClassifier con la profundidad elegida (random_state=SEED).
arbol = evaluar("Árbol de decisión (profundidad ...)", ...)
plt.figure(figsize=(3.2, 3.2))
plt.imshow(arbol.feature_importances_.reshape(28, 28), cmap="Blues"); plt.axis("off")
plt.title("¿Qué píxeles pregunta el árbol?"); plt.show()

## Parte 4 · Modelos generativos: LDA, QDA y Naive Bayes

Estos modelos aprenden **cómo es cada dígito** (una gaussiana por clase) y deciden con la regla de Bayes (M1 §6–§7). Primero en Iris, donde los supuestos son razonables:

In [ ]:
from sklearn.datasets import load_iris
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.model_selection import cross_val_score
from sklearn.naive_bayes import GaussianNB

Xi, yi = load_iris(return_X_y=True)
for nombre, m in [("LDA", LinearDiscriminantAnalysis()), ("QDA", QuadraticDiscriminantAnalysis()), ("Naive Bayes", GaussianNB())]:
    print(f"Iris · {nombre:<12} exactitud CV = {cross_val_score(m, Xi, yi, cv=5).mean():.3f}")

Ahora en MNIST. QDA necesita estimar una matriz de 784 × 784 por dígito; muchos píxeles del borde siempre valen 0 y las matrices son singulares, así que la regularizamos con `reg_param`.

In [ ]:
lda = evaluar("LDA (covarianza compartida)", LinearDiscriminantAnalysis())
qda = evaluar("QDA regularizada (reg_param=0.5)", QuadraticDiscriminantAnalysis(reg_param=0.5))

**Tu turno.** Evalúa Naive Bayes gaussiano. Antes de ejecutar, apuesta: ¿le irá mejor o peor que a LDA? Recuerda que supone que los píxeles son independientes dentro de cada dígito.

In [ ]:
# TODO: evalúa GaussianNB() con la función evaluar.
nb = evaluar("Naive Bayes gaussiano", ...)
assert resultados[-1]["exactitud"] < resultados[-3]["exactitud"], "En MNIST, Naive Bayes debería quedar por debajo de LDA."

Los píxeles vecinos están muy correlacionados: Naive Bayes cuenta varias veces la misma evidencia y se vuelve exageradamente seguro. Mira la confianza media (probabilidad de la clase elegida) de cada modelo:

In [ ]:
for nombre, m in [("LDA", lda), ("Naive Bayes", nb)]:
    p = m.predict_proba(X_te).max(axis=1)
    print(f"{nombre:<12} confianza media = {p.mean():.3f} · exactitud = {m.score(X_te, y_te):.3f}")

## Parte 5 · SVM y k vecinos más cercanos

Dos modelos que dependen de distancias. En MNIST todos los píxeles comparten escala (0 a 1), así que no hace falta estandarizar. La SVM con kernel RBF es la más lenta de entrenar; kNN no entrena, pero cada predicción recorre las 10 000 imágenes guardadas.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

svm = evaluar("SVM con kernel RBF (C=10)", SVC(kernel="rbf", C=10, gamma="scale"))

**Tu turno.** Evalúa kNN con `n_neighbors=3`.

In [ ]:
# TODO: evalúa KNeighborsClassifier(n_neighbors=3).
knn = evaluar("k vecinos (k=3)", ...)

### La tabla final

In [ ]:
tabla = pd.DataFrame(resultados).sort_values("exactitud", ascending=False).reset_index(drop=True)
tabla.loc[len(tabla)] = {"modelo": "Baseline (dígito más frecuente)", "exactitud": acc_base, "entrenar (s)": 0.0, "predecir (s)": 0.0}
tabla.round({"exactitud": 3, "entrenar (s)": 1, "predecir (s)": 2})

In [ ]:
assert tabla.loc[0, "modelo"].startswith(("SVM", "k vecinos")), "Con píxeles crudos, los modelos por distancias suelen ganar."
mejor = svm if tabla.loc[0, "modelo"].startswith("SVM") else knn
pred = mejor.predict(X_te)
errores = np.flatnonzero(pred != y_te)[:12]
fig, axs = plt.subplots(1, len(errores), figsize=(12, 1.6))
for ax, i in zip(axs, errores):
    ax.imshow(X_te[i].reshape(28, 28), cmap="Greys"); ax.set_title(f"{y_te[i]}→{pred[i]}", fontsize=9, color=UNIBE["naranja"]); ax.axis("off")
plt.suptitle(f"Errores de {tabla.loc[0, 'modelo']} (real → predicho)", y=1.1); plt.show()

**Preguntas para la discusión (Clase 1):**

1. ¿Por qué Naive Bayes queda tan atrás en imágenes y, en cambio, funciona bien en Iris?
2. ¿Qué modelo elegirías para una aplicación en un teléfono que debe responder en milisegundos? ¿Y si tuvieras los 60 000 ejemplos?
3. Mira los errores del mejor modelo: ¿los cometería también una persona?

## Reto · Escala realista con Fashion-MNIST

**Fashion-MNIST** tiene el mismo formato que MNIST (28 × 28, 10 clases, 60 000 + 10 000), pero las imágenes son fotos de prendas: camisetas, pantalones, suéteres, vestidos, abrigos, sandalias, camisas, zapatillas, bolsos y botines. Se creó porque MNIST ya era «demasiado fácil». Repite la comparación con tres modelos y mira cuánto cae la exactitud.

In [ ]:
def cargar_fashion():
    """Fashion-MNIST desde el repositorio público de Keras (cuatro archivos en formato IDX comprimido)."""
    def imagenes(nombre):
        with gzip.open(descargar(nombre)) as f:
            return np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 28, 28)

    def etiquetas(nombre):
        with gzip.open(descargar(nombre)) as f:
            return np.frombuffer(f.read(), np.uint8, offset=8)
    return (imagenes("train-images-idx3-ubyte.gz"), etiquetas("train-labels-idx1-ubyte.gz"),
            imagenes("t10k-images-idx3-ubyte.gz"), etiquetas("t10k-labels-idx1-ubyte.gz"))


PRENDAS = ["camiseta", "pantalón", "suéter", "vestido", "abrigo", "sandalia", "camisa", "zapatilla", "bolso", "botín"]
Ftr, fytr, Fte, fyte = cargar_fashion()
F_tr, fy_tr = Ftr[idx_tr].reshape(-1, 784) / 255.0, fytr[idx_tr]
F_te, fy_te = Fte[idx_te].reshape(-1, 784) / 255.0, fyte[idx_te]
fig, axs = plt.subplots(1, 10, figsize=(11, 1.6))
for k, ax in enumerate(axs):
    i = int(np.flatnonzero(fy_tr == k)[0])
    ax.imshow(F_tr[i].reshape(28, 28), cmap="Greys"); ax.set_title(PRENDAS[k], fontsize=8); ax.axis("off")
plt.show()

In [ ]:
# TODO (reto): entrena softmax, LDA y SVM en Fashion-MNIST y compara su exactitud con la de MNIST.
for nombre, m in [...]:
    ...

## Desafío opcional con IA agéntica · Un séptimo clasificador para MNIST

**Objetivo.** Usar un asistente de IA para sumar un modelo a la comparación de MNIST, verificar que el código hace lo que dice y detectar sus errores.

**Prompt inicial.** Úsalo en la herramienta que prefieras (Claude Code, Codex, Gemini en Colab, ChatGPT…), con este notebook resuelto como contexto. Pide primero un plan y revisa cada paso antes de aprobarlo.

```text
En el notebook resuelto L1.2 (MNIST con seis clasificadores, 10 000 imágenes de entrenamiento y 2 000 de prueba), muéstrame primero un plan. Después agrega un séptimo clasificador que no esté en la tabla (por ejemplo, un bosque aleatorio o la regresión logística sobre 50 componentes principales), evalúalo con la misma función evaluar y la misma muestra, y agrega a la tabla una columna con la exactitud de validación cruzada de 3 pliegues calculada solo con el entrenamiento. No cambies la muestra ni la semilla.
```

**Cómo verificar el resultado**

- El nuevo modelo usa exactamente las mismas X_tr, y_tr, X_te, y_te que los demás.
- La validación cruzada se calcula sin la prueba.
- Puedes explicar con tus palabras por qué el nuevo modelo queda por encima o por debajo de la SVM.

**Declara el uso de IA** (como pide el sílabo): herramienta, prompts relevantes, qué verificaste tú y qué corregiste. El desafío es opcional y no se califica; lo que cuenta es que puedas explicar cada decisión.